# Phase 2 — Data Integration

## Objective

The objective of this phase is to integrate the transaction-level information from `train_transaction` with the identity/device information from `train_identity` using `TransactionID`.

Before performing the merge, the relationship between the two datasets will be validated to ensure that the integration does not introduce duplicate records, lose transactions, or create unexpected row multiplication.

Only data integration is performed in this notebook. Preprocessing, feature engineering, modeling, and deployment are not performed.

## Step 1 — Load the datasets

In [1]:
from pathlib import Path
import pandas as pd
import numpy as np

project = Path.cwd().parent
transaction_path = project / "data" / "raw" / "train_transaction.csv"
identity_path = project / "data" / "raw" / "train_identity.csv"

train_transaction = pd.read_csv(transaction_path, low_memory=False)
train_identity = pd.read_csv(identity_path, low_memory=False)

print("Transaction shape:", train_transaction.shape)
print("Identity shape:", train_identity.shape)
print("Original datasets were not modified.")

Transaction shape: (590540, 394)
Identity shape: (144233, 41)
Original datasets were not modified.


**Interpretation:** The original transaction and identity files were loaded as separate DataFrames. No columns were dropped, no missing values were filled, and no encoding or scaling was performed.

## Step 2 — Verify the original dataset sizes

In [2]:
print("Transaction rows:", len(train_transaction))
print("Transaction columns:", len(train_transaction.columns))
print("Identity rows:", len(train_identity))
print("Identity columns:", len(train_identity.columns))

Transaction rows: 590540
Transaction columns: 394
Identity rows: 144233
Identity columns: 41


**Decision:** These values are the baseline against which the merged dataset will be checked.

## Step 3 — Validate TransactionID

In [3]:
print("TransactionID examples in transactions:")
display(train_transaction["TransactionID"].head())
print("TransactionID examples in identity:")
display(train_identity["TransactionID"].head())

print("Transaction TransactionID type:", train_transaction["TransactionID"].dtype)
print("Identity TransactionID type:", train_identity["TransactionID"].dtype)
print("Transaction missing TransactionID:", train_transaction["TransactionID"].isna().sum())
print("Identity missing TransactionID:", train_identity["TransactionID"].isna().sum())
print("Transaction unique TransactionIDs:", train_transaction["TransactionID"].nunique())
print("Identity unique TransactionIDs:", train_identity["TransactionID"].nunique())
print("Transaction duplicate IDs:", train_transaction["TransactionID"].duplicated().sum())
print("Identity duplicate IDs:", train_identity["TransactionID"].duplicated().sum())

TransactionID examples in transactions:


0    2987000
1    2987001
2    2987002
3    2987003
4    2987004
Name: TransactionID, dtype: int64

TransactionID examples in identity:


0    2987004
1    2987008
2    2987010
3    2987011
4    2987016
Name: TransactionID, dtype: int64

Transaction TransactionID type: int64
Identity TransactionID type: int64
Transaction missing TransactionID: 0
Identity missing TransactionID: 0


Transaction unique TransactionIDs: 590540
Identity unique TransactionIDs: 144233


Transaction duplicate IDs: 0


Identity duplicate IDs: 0


**Interpretation:** The key has the same data type in both datasets, has no missing values, and is unique in each dataset. It is therefore suitable for a controlled join. No type conversion is needed.

## Step 4 — Understand the relationship without merging

In [4]:
transaction_ids = set(train_transaction["TransactionID"])
identity_ids = set(train_identity["TransactionID"])
common_ids = transaction_ids.intersection(identity_ids)

relationship = pd.DataFrame({
    "Metric": [
        "Transaction Records", "Identity Records", "Unique Transaction IDs",
        "Unique Identity IDs", "Common Transaction IDs",
        "Transactions Without Identity", "Identity Records Without Transaction",
        "Identity Coverage %"
    ],
    "Value": [
        len(train_transaction), len(train_identity), len(transaction_ids),
        len(identity_ids), len(common_ids), len(transaction_ids - identity_ids),
        len(identity_ids - transaction_ids), len(common_ids) / len(transaction_ids) * 100
    ]
})
display(relationship)

,Metric,Value
0,Transaction Records,590540.000000
1,Identity Records,144233.000000
2,Unique Transaction IDs,590540.000000
3,Unique Identity IDs,144233.000000
4,Common Transaction IDs,144233.000000
5,Transactions Without Identity,446307.000000
6,Identity Records Without Transaction,0.000000
7,Identity Coverage %,24.423917


**Join decision:** A **left join** from `train_transaction` to `train_identity` is appropriate. It keeps every transaction, including transactions without identity information. This is important because `isFraud` belongs to the transaction dataset and no labeled transaction should be lost.

## Step 5 — Perform the merge

In [5]:
merged_data = train_transaction.merge(
    train_identity,
    on="TransactionID",
    how="left",
    validate="one_to_one",
    indicator=True
)

merge_status = merged_data["_merge"].value_counts()
merged_data = merged_data.drop(columns=["_merge"])
print("Merged shape:", merged_data.shape)
print("Merge status:")
display(merge_status)

C:\Users\HP\AppData\Local\Temp\ipykernel_11788\3241652034.py:1: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  merged_data = train_transaction.merge(


Merged shape: (590540, 434)
Merge status:


C:\Users\HP\AppData\Local\Temp\ipykernel_11788\3241652034.py:1: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  merged_data = train_transaction.merge(


_merge
left_only     446307
both          144233
right_only         0
Name: count, dtype: int64

**Interpretation:** The temporary merge indicator shows which transactions received identity information and which remained transaction-only. It is removed from the final integrated dataset.

## Step 6 — Check for row multiplication and transaction loss

In [6]:
print("Original transaction rows:", len(train_transaction))
print("Merged rows:", len(merged_data))
print("Rows lost:", len(train_transaction) - len(merged_data))
print("Unexpected row multiplication:", len(merged_data) > len(train_transaction))

original_ids = set(train_transaction["TransactionID"])
merged_ids = set(merged_data["TransactionID"])
print("Transactions missing after merge:", len(original_ids - merged_ids))

Original transaction rows: 590540
Merged rows: 590540
Rows lost: 0
Unexpected row multiplication: False


Transactions missing after merge: 0


**Decision:** The merge must preserve one row per transaction. If the merged row count were larger, duplicate identity keys would need investigation. If it were smaller, transaction loss would require investigation.

## Step 7 — Verify TransactionID and target preservation

In [7]:
print("Merged unique TransactionIDs:", merged_data["TransactionID"].nunique())
print("Merged duplicate TransactionIDs:", merged_data["TransactionID"].duplicated().sum())
print("isFraud exists after merge:", "isFraud" in merged_data.columns)

print("Original target counts:")
display(train_transaction["isFraud"].value_counts().sort_index())
print("Merged target counts:")
display(merged_data["isFraud"].value_counts().sort_index())
print("Target preserved:", train_transaction["isFraud"].value_counts().sort_index().equals(merged_data["isFraud"].value_counts().sort_index()))

Merged unique TransactionIDs: 590540
Merged duplicate TransactionIDs: 0
isFraud exists after merge: True
Original target counts:


isFraud
0    569877
1     20663
Name: count, dtype: int64

Merged target counts:


isFraud
0    569877
1     20663
Name: count, dtype: int64

Target preserved: True


## Step 8 — Verify identity coverage and missingness after merge

In [8]:
merged_identity_rows = merged_data["TransactionID"].isin(identity_ids).sum()
print("Transactions with Identity:", merged_identity_rows)
print("Transactions without Identity:", len(merged_data) - merged_identity_rows)
print("Identity Coverage %:", merged_identity_rows / len(merged_data) * 100)

identity_columns = [c for c in train_identity.columns if c != "TransactionID"]
identity_missing = merged_data[identity_columns].isna().mean().sort_values(ascending=False) * 100
print("Top Identity missing percentages after merge:")
display(identity_missing.head(10).to_frame("Missing %"))

Transactions with Identity: 144233
Transactions without Identity: 446307
Identity Coverage %: 24.42391709283029


Top Identity missing percentages after merge:


,Missing %
id_24,99.196159
id_25,99.130965
id_07,99.127070
id_08,99.127070
id_21,99.126393
id_26,99.125715
id_27,99.124699
id_22,99.124699
id_23,99.124699
id_18,92.360721


**Interpretation:** Missing identity values for transaction-only records are expected. They are not filled and those transactions are not removed.

## Step 9 — Check column structure and data types

In [9]:
print("Total merged columns:", len(merged_data.columns))
print("Transaction columns:", len(train_transaction.columns))
print("Identity columns excluding TransactionID:", len(identity_columns))
print("Duplicate column names:", merged_data.columns.duplicated().sum())
print("Columns with _x or _y suffix:", [c for c in merged_data.columns if c.endswith("_x") or c.endswith("_y")])
print("Merged data types:")
display(merged_data.dtypes.value_counts())

Total merged columns: 434
Transaction columns: 394
Identity columns excluding TransactionID: 40
Duplicate column names: 0
Columns with _x or _y suffix: []
Merged data types:


float64    399
str         31
int64        4
Name: count, dtype: int64

## Step 10 — Basic sanity check

In [10]:
print("First rows of merged data:")
display(merged_data.head())

matched_example_id = next(iter(common_ids))
unmatched_example_id = next(iter(transaction_ids - identity_ids))
print("Matched transaction example:")
display(merged_data[merged_data["TransactionID"] == matched_example_id].head(1))
print("Transaction without identity example:")
display(merged_data[merged_data["TransactionID"] == unmatched_example_id].head(1))

First rows of merged data:


,TransactionID,isFraud,TransactionDT,TransactionAmt,ProductCD,card1,card2,card3,card4,card5,...,id_31,id_32,id_33,id_34,id_35,id_36,id_37,id_38,DeviceType,DeviceInfo
0,2987000,0,86400,68.5,W,13926,NaN,150.0,discover,142.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,2987001,0,86401,29.0,W,2755,404.0,150.0,mastercard,102.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,2987002,0,86469,59.0,W,4663,490.0,150.0,visa,166.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,2987003,0,86499,50.0,W,18132,567.0,150.0,mastercard,117.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,2987004,0,86506,50.0,H,4497,514.0,150.0,mastercard,102.0,...,samsung browser 6.2,32.0,2220x1080,match_status:2,T,F,T,T,mobile,SAMSUNG SM-G892A Build/NRD90M


Matched transaction example:


,TransactionID,isFraud,TransactionDT,TransactionAmt,ProductCD,card1,card2,card3,card4,card5,...,id_31,id_32,id_33,id_34,id_35,id_36,id_37,id_38,DeviceType,DeviceInfo
420877,3407877,0,10660557,10.392,C,13413,103.0,185.0,visa,137.0,...,other,NaN,NaN,NaN,F,F,T,F,mobile,SM-G610M Build/MMB29K


Transaction without identity example:


,TransactionID,isFraud,TransactionDT,TransactionAmt,ProductCD,card1,card2,card3,card4,card5,...,id_31,id_32,id_33,id_34,id_35,id_36,id_37,id_38,DeviceType,DeviceInfo
0,2987000,0,86400,68.5,W,13926,NaN,150.0,discover,142.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


## Step 11 — Final integration validation

In [11]:
validation = pd.DataFrame({
    "Check": [
        "Transaction rows preserved", "TransactionID preserved", "No unexpected row multiplication",
        "Target preserved", "Identity records attached", "Transactions without Identity retained",
        "No unexpected duplicate IDs", "Column structure verified", "Data types inspected"
    ],
    "Result": [
        len(merged_data) == len(train_transaction),
        "TransactionID" in merged_data.columns,
        len(merged_data) <= len(train_transaction),
        train_transaction["isFraud"].value_counts().sort_index().equals(merged_data["isFraud"].value_counts().sort_index()),
        len(common_ids) > 0,
        len(transaction_ids - identity_ids) > 0,
        merged_data["TransactionID"].duplicated().sum() == 0,
        len(merged_data.columns) == len(train_transaction.columns) + len(train_identity.columns) - 1,
        merged_data.dtypes.value_counts().sum() == len(merged_data.columns)
    ]
})
display(validation)
print("All validation checks passed:", validation["Result"].all())

,Check,Result
0,Transaction rows preserved,True
1,TransactionID preserved,True
2,No unexpected row multiplication,True
3,Target preserved,True
4,Identity records attached,True
5,Transactions without Identity retained,True
6,No unexpected duplicate IDs,True
7,Column structure verified,True
8,Data types inspected,True


All validation checks passed: True


## Step 12 — Save the integrated dataset

In [12]:
processed_folder = project / "data" / "processed"
processed_folder.mkdir(exist_ok=True)
output_path = processed_folder / "merged_data.csv.gz"
merged_data.to_csv(output_path, index=False, compression="gzip")
print("Saved:", output_path)
print("File size in MB:", round(output_path.stat().st_size / 1024**2, 2))

Saved: C:\Users\HP\Desktop\fraud-detection-system\data\processed\merged_data.csv.gz
File size in MB: 47.95


A single compressed CSV file is used because the dataset is large and PostgreSQL is a later approved phase. No additional uncompressed copy is created.

## Phase 2 — Data Integration Final Report

In [13]:
print("1. Original Dataset Sizes")
print("train_transaction:", train_transaction.shape)
print("train_identity:", train_identity.shape)
print("\n2. TransactionID Validation")
print("Missing IDs: transaction =", train_transaction["TransactionID"].isna().sum(), ", identity =", train_identity["TransactionID"].isna().sum())
print("Duplicate IDs: transaction =", train_transaction["TransactionID"].duplicated().sum(), ", identity =", train_identity["TransactionID"].duplicated().sum())
print("Unique IDs: transaction =", train_transaction["TransactionID"].nunique(), ", identity =", train_identity["TransactionID"].nunique())
print("\n3. Dataset Relationship")
print("Common IDs:", len(common_ids))
print("Transactions without Identity:", len(transaction_ids - identity_ids))
print("Identity coverage:", round(len(common_ids) / len(transaction_ids) * 100, 6), "%")
print("\n4. Merge Strategy")
print("Selected Join: LEFT JOIN")
print("Reason: preserve every labeled transaction, including those without Identity information.")
print("\n5. Final Integrated Dataset")
print("Rows:", len(merged_data))
print("Columns:", len(merged_data.columns))
print("Unique TransactionIDs:", merged_data["TransactionID"].nunique())
print("Duplicate TransactionIDs:", merged_data["TransactionID"].duplicated().sum())
print("\n6. Target Validation")
print("Fraud Count:", int((merged_data["isFraud"] == 1).sum()))
print("Legitimate Count:", int((merged_data["isFraud"] == 0).sum()))
print("Target Preserved: True")
print("\n7. Data Integrity")
print("Rows Lost:", len(train_transaction) - len(merged_data))
print("Rows Duplicated:", max(0, len(merged_data) - len(train_transaction)))
print("Unexpected Issues: None detected")
print("\n8. Identity Information")
print("Transactions with Identity:", len(common_ids))
print("Transactions without Identity:", len(transaction_ids - identity_ids))
print("\n9. Final Decision")
print("merged_data is ready for the next phase after review.")
print("Missing Value Imputation: NOT YET")
print("Encoding: NOT YET")
print("Scaling: NOT YET")
print("Feature Selection: NOT YET")
print("Feature Engineering: NOT YET")
print("SMOTE: NOT YET")
print("Model Training: NOT YET")

1. Original Dataset Sizes
train_transaction: (590540, 394)
train_identity: (144233, 41)

2. TransactionID Validation
Missing IDs: transaction = 0 , identity = 0
Duplicate IDs: transaction = 0 , identity = 0


Unique IDs: transaction = 590540 , identity = 144233

3. Dataset Relationship
Common IDs: 144233
Transactions without Identity: 446307
Identity coverage: 24.423917 %

4. Merge Strategy
Selected Join: LEFT JOIN
Reason: preserve every labeled transaction, including those without Identity information.

5. Final Integrated Dataset
Rows: 590540
Columns: 434


Unique TransactionIDs: 590540
Duplicate TransactionIDs: 0

6. Target Validation
Fraud Count: 20663
Legitimate Count: 569877
Target Preserved: True

7. Data Integrity
Rows Lost: 0
Rows Duplicated: 0
Unexpected Issues: None detected

8. Identity Information
Transactions with Identity: 144233
Transactions without Identity: 446307

9. Final Decision
merged_data is ready for the next phase after review.
Missing Value Imputation: NOT YET
Encoding: NOT YET
Scaling: NOT YET
Feature Selection: NOT YET
Feature Engineering: NOT YET
SMOTE: NOT YET
Model Training: NOT YET


**STOP:** Phase 2 is complete. Do not start `03_eda.ipynb`, preprocessing, PostgreSQL, API, dashboard, or model training until explicit approval.